[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-08-bm25-and-retrieval-metrics.ipynb)

# Advanced Discussion 8 — BM25 from scratch and retrieval metrics

BM25 and an inverted index in about fifty lines, recall/MRR/nDCG from scratch, and a first comparison on the SciFact collection.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Downloads BEIR SciFact (a few MB) from the Hugging Face hub.

In [ ]:
%pip install -q datasets rank-bm25 scikit-learn

In [ ]:
import re, math, time, numpy as np
from collections import Counter
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
np.random.seed(0)

## 1. data: BEIR SciFact (5,183 abstracts; 300 test claims with relevance judgments)

In [ ]:
corpus = load_dataset("BeIR/scifact", "corpus", split="corpus"); queries = load_dataset("BeIR/scifact", "queries", split="queries")
qrels_ds = load_dataset("BeIR/scifact-qrels", split="test")
doc_ids = [d["_id"] for d in corpus]; doc_text = [(d["title"] + ". " + d["text"]) for d in corpus]
qtext = {q["_id"]: q["text"] for q in queries}
qrels = {}
for r in qrels_ds:
    if r["score"] > 0: qrels.setdefault(str(r["query-id"]), {})[str(r["corpus-id"])] = r["score"]
test_q = [q for q in qrels if q in qtext]
print("docs", len(doc_ids), "test queries", len(test_q), "avg relevant per query %.2f" % np.mean([len(qrels[q]) for q in test_q]))

## 2. metrics from scratch

In [ ]:
def recall_at_k(ranked, rel, k): return len(set(ranked[:k]) & set(rel)) / len(rel)
def mrr_at_k(ranked, rel, k):
    for i, d in enumerate(ranked[:k], 1):
        if d in rel: return 1.0 / i
    return 0.0
def ndcg_at_k(ranked, rel, k):
    dcg = sum((2 ** rel.get(d, 0) - 1) / math.log2(i + 1) for i, d in enumerate(ranked[:k], 1))
    ideal = sorted(rel.values(), reverse=True)[:k]
    idcg = sum((2 ** g - 1) / math.log2(i + 1) for i, g in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0
def evaluate(run, name):
    m = {"nDCG@10": np.mean([ndcg_at_k(run[q], qrels[q], 10) for q in test_q]), "MRR@10": np.mean([mrr_at_k(run[q], qrels[q], 10) for q in test_q]),
         "Recall@10": np.mean([recall_at_k(run[q], qrels[q], 10) for q in test_q]), "Recall@100": np.mean([recall_at_k(run[q], qrels[q], 100) for q in test_q])}
    print("%-34s nDCG@10 %.3f  MRR@10 %.3f  Recall@10 %.3f  Recall@100 %.3f" % (name, *m.values())); return m
# tiny worked example
ranked = ["d3", "d1", "d7", "d2", "d9"]; rel = {"d1": 1, "d2": 1, "d5": 1}
print("worked: recall@3 %.3f, MRR %.3f, nDCG@5 %.3f" % (recall_at_k(ranked, rel, 3), mrr_at_k(ranked, rel, 5), ndcg_at_k(ranked, rel, 5)))

## 3. BM25 from scratch, checked against a library

In [ ]:
tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b; self.docs = [Counter(tok(d)) for d in docs]; self.len = np.array([sum(c.values()) for c in self.docs]); self.avg = self.len.mean()
        df = Counter(t for c in self.docs for t in c); N = len(docs)
        self.idf = {t: math.log((N - n + 0.5) / (n + 0.5) + 1) for t, n in df.items()}
        self.inv = {}
        for i, c in enumerate(self.docs):
            for t, f in c.items(): self.inv.setdefault(t, []).append((i, f))
    def scores(self, query):
        s = np.zeros(len(self.docs))
        for t in set(tok(query)):
            if t not in self.idf: continue
            for i, f in self.inv[t]:
                s[i] += self.idf[t] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return s
t0 = time.time(); bm = BM25(doc_text); print("index built in %.1fs" % (time.time() - t0))
lib = BM25Okapi([tok(d) for d in doc_text])
q0 = qtext[test_q[0]]; print("query:", q0)
top_mine = np.argsort(-bm.scores(q0))[:5]; top_lib = np.argsort(-lib.get_scores(tok(q0)))[:5]
print("top-5 identical to rank_bm25:", list(top_mine) == list(top_lib), "(mine %s | library %s)" % (list(top_mine), list(top_lib)))
run_bm25 = {q: [doc_ids[i] for i in np.argsort(-bm.scores(qtext[q]))[:100]] for q in test_q}
tf = TfidfVectorizer(sublinear_tf=True, stop_words="english").fit(doc_text); D = tf.transform(doc_text)
run_tfidf = {q: [doc_ids[i] for i in np.argsort(-(D @ tf.transform([qtext[q]]).T).toarray().ravel())[:100]] for q in test_q}
res = {}
res["tfidf"] = evaluate(run_tfidf, "TF-IDF (cosine)"); res["bm25"] = evaluate(run_bm25, "BM25 (from scratch)")

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.